## Setup
Import all necessary libraries and define file paths to the datasets stored in Unity Catalog Volumes.

In [0]:
from pyspark.sql.functions import col, year, month, avg, max, min, count
import matplotlib.pyplot as plt
import pandas as pd

base_path = "/Volumes/workspace/default/datasets"

hourly_path = f"{base_path}/all_buoys_hourly_data_new.parquet"
metadata_path = f"{base_path}/buoy_metadata_in_water.csv"

## Data Ingestion
Load the hourly buoy readings and buoy metadata from storage. The main dataset is stored in Parquet format, which unlike CSV natively stores schema and data type information.

The dataset contains the following key fields:
- buoy_id: Unique identifier for the buoy station (e.g., '46050').
- datetime_utc: Timestamp of the measurement in Coordinated Universal Time (UTC).
- water_temp_c: Sea surface temperature in degrees Celsius (°C).
- wave_height_m: Significant wave height in meters (m).
- latitude: Latitude of the buoy in decimal degrees.
- longitude Longitude of the buoy in decimal degrees.

Note: The dataset additionally contains air temperature, wind speed, atmospheric 
pressure, and pre-engineered climate features which are reserved for future analysis.

In [0]:
# Load the parquet file
df = spark.read.parquet(hourly_path)
df_metadata = spark.read.csv(metadata_path, header=True, inferSchema=True)

##Register DataFrames as Temporary Views

In [0]:
df.createOrReplaceTempView("buoys")
df_metadata.createOrReplaceTempView("metadata")

## Feature Engineering: Ocean Region
Assign each buoy reading to an ocean region based on its latitude and longitude coordinates. This enriches the dataset for regional analysis. Note: the Southern Ocean is excluded as the dataset contains no buoy readings below -55° latitude.

In [0]:
from pyspark.sql.functions import when

df = df.withColumn("ocean_region",
    # Southern Ocean - Excluded. Dataset contains no buoys below -55° latitude.
    # Arctic Ocean
    when(col("latitude") > 66.56, "Arctic Ocean")
    # Indian Ocean
    .when(
        (col("latitude").between(-55, 31.18)) & 
        (col("longitude").between(20, 146.9)),
        "Indian Ocean"
    )
    # North Atlantic
    .when(
        (col("latitude").between(-0.936, 68.64)) & 
        (col("longitude").between(-98.05, 12)),
        "North Atlantic"
    )
    # South Atlantic
    .when(
        (col("latitude").between(-55, 0.075)) & 
        (col("longitude").between(-69.6, 20)),
        "South Atlantic"
    )
    # North Pacific - wraps around 180° line
    .when(
        (col("latitude").between(0, 66.56)) & 
        ((col("longitude") >= 117.5) | (col("longitude") <= -76.98)),
        "North Pacific"
    )
    # South Pacific - wraps around 180° line
    .when(
        (col("latitude").between(-55, 3.4)) & 
        ((col("longitude") >= 130.11) | (col("longitude") <= -67.27)),
        "South Pacific"
    )
    .otherwise("Unknown")
)

# Updating view with ocean regions.

df.createOrReplaceTempView("buoys")

## Analysis: Global Sea Surface Temperature Trend (1999-2025)
Plot the average annual sea surface temperature across all buoys to identify long term warming trends. A linear trend line is included to highlight the overall direction of change over time.

In [0]:
from pyspark.sql.functions import year, round

df_temp_trend = df.filter(col("water_temp_c").isNotNull()) \
    .groupBy(year("datetime_utc").alias("year")) \
    .agg(round(avg("water_temp_c"), 4).alias("avg_water_temp")) \
    .orderBy("year")

# Convert to pandas for plotting
temp_trend_pd = df_temp_trend.toPandas()

plt.figure(figsize=(14, 6))
plt.plot(temp_trend_pd["year"], temp_trend_pd["avg_water_temp"], 
         color="steelblue", linewidth=2, marker="o", markersize=4)

# Add a trend line
import numpy as np
z = np.polyfit(temp_trend_pd["year"], temp_trend_pd["avg_water_temp"], 1)
p = np.poly1d(z)
plt.plot(temp_trend_pd["year"], p(temp_trend_pd["year"]), 
         color="red", linewidth=1.5, linestyle="--", label="Trend")

plt.xlabel("Year")
plt.ylabel("Average Sea Surface Temperature (°C)")
plt.title("Global Average Sea Surface Temperature (1999-2024)")
plt.legend()
plt.grid(True, alpha=0.3)
plt.tight_layout()
plt.show()

## Analysis: Active Buoy Coverage by Year (1999-2025)
Plot the number of distinct active buoys per year to assess data reliability over time. 
A vertical reference line marks 2008, the point at which coverage reached approximately 
50 buoys - enough geographic spread to begin drawing meaningful global insights from the data.

In [0]:
from pyspark.sql.functions import year, countDistinct

df_buoy_coverage = df.groupBy(year("datetime_utc").alias("year")) \
    .agg(countDistinct("buoy_id").alias("active_buoys")) \
    .orderBy("year")

# Convert to pandas for plotting
buoy_coverage_pd = df_buoy_coverage.toPandas()

plt.figure(figsize=(14, 6))
plt.plot(buoy_coverage_pd["year"], buoy_coverage_pd["active_buoys"],
         color="steelblue", linewidth=2, marker="o", markersize=4)

plt.xlabel("Year")
plt.ylabel("Number of Active Buoys")
plt.title("Number of Active Buoys by Year (1999-2025)")
plt.axvline(x=2008, color="red", linewidth=1.5,
            linestyle="--", label="Coverage becomes reliable (~50 buoys)")
plt.legend()
plt.grid(True, alpha=0.3)
plt.tight_layout()
plt.show()

## Analysis: Rogue Wave Detection
A rogue wave is traditionally defined as a wave more than twice the significant wave 
height of surrounding seas. 

Wave height thresholds are calculated per ocean region, since normal wave conditions 
vary significantly by location. Any reading exceeding 3 standard deviations above 
that region's mean is classified as a rogue wave event.

### Threshold Calculation

In [0]:
from pyspark.sql.functions import stddev, mean

# Calculate mean and stddev per region
wave_stats_by_region = df.filter(col("wave_height_m").isNotNull()) \
    .groupBy("ocean_region") \
    .agg(
        mean("wave_height_m").alias("mean_wave"),
        stddev("wave_height_m").alias("stddev_wave")
    ) \
    .withColumn("threshold", col("mean_wave") + (3 * col("stddev_wave")))

display(wave_stats_by_region)

### Rogue Wave Detection and Regional Aggregation

In [0]:
df_rogue = df.filter(col("wave_height_m").isNotNull()) \
    .join(wave_stats_by_region, on="ocean_region") \
    .filter(col("wave_height_m") >= col("threshold")) \
    .select(
        "buoy_id",
        "datetime_utc",
        "wave_height_m",
        "water_temp_c",
        "wind_speed_ms",
        "latitude",
        "longitude",
        "ocean_region",
        "threshold"
    ).orderBy(col("wave_height_m").desc())

df_rogue_by_region = df_rogue.groupBy("ocean_region") \
    .agg(
        count("wave_height_m").alias("rogue_wave_count"),
        round(avg("wave_height_m"), 2).alias("avg_rogue_height"),
        round(max("wave_height_m"), 2).alias("max_rogue_height")
    ).orderBy(col("rogue_wave_count").desc())

# Get buoy count and total readings per region
buoy_count_by_region = df.groupBy("ocean_region") \
    .agg(
        countDistinct("buoy_id").alias("buoy_count"),
        count("buoy_id").alias("total_readings")
    )

# Join into the rogue wave regional summary
df_rogue_enriched = df_rogue_by_region \
    .join(buoy_count_by_region, on="ocean_region") \
    .select(
        "ocean_region",
        "buoy_count",
        "total_readings",
        "rogue_wave_count",
        "avg_rogue_height",
        "max_rogue_height"
    ).orderBy(col("rogue_wave_count").desc())

display(df_rogue_enriched)

### Results: Rogue Wave Events by Ocean Region

In [0]:
rogue_region_pd = df_rogue_enriched.toPandas()

plt.figure(figsize=(12, 6))
bars = plt.bar(rogue_region_pd["ocean_region"], 
               rogue_region_pd["rogue_wave_count"],
               color="steelblue")

# Add value labels on top of each bar
for bar, val in zip(bars, rogue_region_pd["rogue_wave_count"]):
    plt.text(bar.get_x() + bar.get_width()/2, bar.get_height() + 10,
             f"{val:,}", ha="center", va="bottom", fontsize=10)

plt.xlabel("Ocean Region")
plt.ylabel("Number of Rogue Wave Events")
plt.title("Rogue Wave Events by Ocean Region")
plt.grid(True, alpha=0.3, axis="y")
plt.tight_layout()
plt.show()